# Burger patty weights: mean and standard deviation chart

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Xbar-s Control Chart** · a worked example from the StatNex teaching guide

**The story.** A burger chain's kitchen weighs five patties a day for 25 working days. Patties should weigh about 113 g. On day 16 the scale at the patty press was set wrongly. This chart measures each day's spread with the standard deviation instead of the range.

**The question.** Do we reach the same conclusion when we measure the spread with the standard deviation?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/xbar_s_chart.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Date,Group number,09:00,11:00,13:00,15:00,17:00
2025-07-01,1.0,112.6,113.0,113.3,112.9,111.8
2025-07-02,2.0,112.8,112.5,113.8,116.1,114.3
2025-07-03,3.0,113.9,114.2,114.2,111.4,113.4
2025-07-04,4.0,112.3,111.8,113.0,114.2,114.7
2025-07-07,5.0,111.5,113.0,113.1,114.5,112.4
2025-07-08,6.0,112.1,111.2,113.7,115.3,113.7
2025-07-09,7.0,113.0,111.1,111.8,110.1,113.1
2025-07-10,8.0,114.4,111.6,113.8,111.5,112.5
2025-07-11,9.0,114.1,113.5,113.5,114.3,113.9
2025-07-14,10.0,113.7,113.5,113.8,113.6,114.9
2025-07-15,11.0,112.2,114.9,112.2,114.1,112.0
2025-07-16,12.0,112.4,113.9,112.4,115.0,112.1
2025-07-17,13.0,115.0,114.1,111.6,113.7,111.1
2025-07-18,14.0,111.1,111.8,116.2,113.2,114.5
2025-07-21,15.0,112.2,110.7,112.8,113.5,113.8
2025-07-22,16.0,115.4,115.2,114.8,119.1,114.7
2025-07-23,17.0,113.6,112.9,113.5,112.0,113.7
2025-07-24,18.0,112.7,112.1,111.3,116.5,111.6
2025-07-25,19.0,113.5,113.1,112.4,113.0,113.4
2025-07-28,20.0,116.0,111.8,110.6,112.8,111.7
2025-07-29,21.0,111.3,112.6,110.9,113.2,110.8
2025-07-30,22.0,112.9,112.0,113.0,113.0,112.5
2025-07-31,23.0,109.8,115.9,111.0,113.0,110.3
2025-08-01,24.0,114.0,110.8,112.3,113.7,112.0
2025-08-04,25.0,113.2,114.3,114.0,112.7,113.6
""")

## Helper: control_signals

A step that no library does, written out. In the code download it is the file `control_signals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
# draw and pattern_signals: defined in the helper cell control_signals above (control_signals.py in the code download)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One row per subgroup (a small sample taken at one time), one column per measurement in
# it (2 to 25 columns). A blank or text cell would change the subgroup's size, so it
# stops the chart instead of being skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["09:00", "11:00", "13:00", "15:00", "17:00"]
values = data[columns].apply(pd.to_numeric, errors="coerce")
if values.isna().any().any():
    raise ValueError("Every cell must hold a number: complete or remove the subgroup.")
values.index = np.arange(1, len(values) + 1)  # subgroup numbers 1, 2, ...
size = len(columns)

## Control-chart constants (JIS Z 9021)

In [ ]:
# Both charts put their limits 3 standard deviations either side of the centre line. The
# process standard deviation is estimated from s-bar, the mean of the subgroup standard
# deviations (it is s-bar / c4, because a sample SD runs a little low on average), and
# these constants turn s-bar into the limits for subgroups of n readings:
#   Xbar chart: CL +/- A3 x s-bar, where A3 = 3 / (c4 sqrt(n))
#   s chart:    UCL = B4 x s-bar and LCL = B3 x s-bar, where B4, B3 = 1 +/- 3 sqrt(1 - c4^2) / c4
# (B3 is 0 where the formula would go below 0). They are the printed table's
# three-decimal values (JIS Z 9021, ASTM, Montgomery's appendix), as StatNex uses them,
# for n = 2..25.
A3 = {2: 2.659, 3: 1.954, 4: 1.628, 5: 1.427, 6: 1.287, 7: 1.182, 8: 1.099, 9: 1.032,
      10: 0.975, 11: 0.927, 12: 0.886, 13: 0.850, 14: 0.817, 15: 0.789, 16: 0.763,
      17: 0.739, 18: 0.718, 19: 0.698, 20: 0.680, 21: 0.663, 22: 0.647, 23: 0.633,
      24: 0.619, 25: 0.606}[size]
B3 = {2: 0.000, 3: 0.000, 4: 0.000, 5: 0.000, 6: 0.030, 7: 0.118, 8: 0.185, 9: 0.239,
      10: 0.284, 11: 0.321, 12: 0.354, 13: 0.382, 14: 0.406, 15: 0.428, 16: 0.448,
      17: 0.466, 18: 0.482, 19: 0.497, 20: 0.510, 21: 0.523, 22: 0.534, 23: 0.545,
      24: 0.555, 25: 0.565}[size]
B4 = {2: 3.267, 3: 2.568, 4: 2.266, 5: 2.089, 6: 1.970, 7: 1.882, 8: 1.815, 9: 1.761,
      10: 1.716, 11: 1.679, 12: 1.646, 13: 1.618, 14: 1.594, 15: 1.572, 16: 1.552,
      17: 1.534, 18: 1.518, 19: 1.503, 20: 1.490, 21: 1.477, 22: 1.466, 23: 1.455,
      24: 1.445, 25: 1.435}[size]

## Points: subgroup mean and standard deviation

In [ ]:
# The mean of each subgroup, and its sample standard deviation (divisor n - 1).
mean = values.mean(axis=1)
spread = values.std(axis=1, ddof=1)


def limits(keep):
    """Centre line, upper and lower limits of each chart, from the subgroups in keep."""
    x_bar, s_bar = mean[keep].mean(), spread[keep].mean()
    return {"mean": (x_bar, x_bar + A3 * s_bar, x_bar - A3 * s_bar),
            "sd": (s_bar, B4 * s_bar, B3 * s_bar)}


keep = pd.Series(True, index=values.index)

## Control chart data

In [ ]:
# One row per subgroup: its mean against the Xbar chart's centre line (CL) and limits
# (UCL, LCL), then its standard deviation against the s chart's. A point outside its
# limits is a signal that something other than everyday variation affected that
# subgroup. Read the s chart first: the Xbar limits are built from s-bar, so they only
# mean something while the spread is stable. n <= 5 makes B3 zero: the s chart then has
# no lower limit.
lines = limits(keep)
table = pd.DataFrame({"mean": mean[keep]})
table["CL"], table["UCL"], table["LCL"] = lines["mean"]
table["SD"] = spread[keep]
table["s CL"], table["s UCL"], table["s LCL"] = lines["sd"]
print()
print(table)
outside_now = ((table["mean"] > table["UCL"]) | (table["mean"] < table["LCL"])
               | (table["SD"] > table["s UCL"]) | (table["SD"] < table["s LCL"]))
print(f"\nSubgroups outside the limits: {list(table.index[outside_now])}")

## The charts

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row. The limits assume a stable process whose subgroup means are roughly
# normal; then a point outside happens by chance about 3 times in 1,000.
for message in (pattern_signals("Xbar chart", table["mean"], table["CL"])
                + pattern_signals("s chart", table["SD"], table["s CL"])):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/xbar_chart.png", "Xbar chart", table["mean"], table["CL"], table["UCL"], table["LCL"])
draw("figures/s_chart.png", "s chart", table["SD"], table["s CL"], table["s UCL"], table["s LCL"])

## What the result says

This chart measures each day's spread with the standard deviation (s), the typical distance of the five weights from their mean, instead of the range. The control limits, which mark the reach of ordinary chance variation, run from 111.3 g to 114.9 g for the daily mean.

Day 16 has a mean of 115.84 g, above the upper control limit (UCL). On the s chart the centre line is 1.25 g and the UCL is 2.61 g. Day 16 has s = 1.84 g, and the largest value, 2.50 g on day 23, is also inside.

So the verdict is the same as with the range: only day 16 stands out, and its patties were heavy, not uneven. With five patties a day, the simpler range loses almost nothing.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.